# Bug 9 PoC - DOM XSS in warninglabelrenderer.js (Lines 119, 140)

**Type:** DOM XSS  
**Severity:** High  
**File:** `profiler-plugin/frontend/lib/warninglabelrenderer.js`  
**Trust Required:** No - fires on untrusted notebooks  

## Vulnerability

`contentChangedHook()` reads `output.data['text/html']` from cell outputs and passes it
directly to jQuery's `$()` at line 119:

```js
const htmlNode = $(output.data['text/html']);  // line 119 - jQuery parses raw HTML into DOM
```

jQuery instantiates DOM nodes immediately, firing event handlers like `onerror`.
This bypasses JupyterLab's notebook trust/sanitization entirely because the plugin
processes the HTML itself rather than going through the built-in renderer.

The parsed HTML is then written back at line 140:

```js
'text/html': ['<table>' + htmlNode.html() + '</table>']  // line 140
```

## Trigger

Open this notebook. The profiler plugin's `contentChangedHook` detects the YARN
table pattern (`hasYarnTable` checks for 'YARN' and '.emr-proxy-link' in the output)
and calls `saveYarnTable()` which parses the HTML via jQuery.

In [1]:
# This cell has pre-populated text/html output simulating SparkMagic YARN table output.
# When the profiler plugin's contentChangedHook fires on notebook open,
# hasYarnTable() matches on 'YARN' and '.emr-proxy-link' in the HTML string,
# then saveYarnTable() calls $(output.data['text/html']) which jQuery-parses
# the <img src=x onerror=...> into a DOM node, triggering the XSS.
#
# No cell execution needed. No notebook trust needed.

YARN Application ID,Kind,State,Spark UI,Driver log
application_1234567890123_0001,spark,idle,Spark UI,Link


In [2]:
# Variant 2: XSS payload inside the emr-proxy-link element.
# This variant targets line 140 specifically. After jQuery parses the HTML (line 119),
# saveYarnTable() reconstructs the output via htmlNode.html() and writes it back:
#   output.setData({ data: { 'text/html': ['<table>' + htmlNode.html() + '</table>'] } })
# The malicious HTML persists in the cell output data, re-triggering on every
# subsequent notebook open or cell change event.

YARN Application ID,Kind,State,Spark UI,Driver log
application_9876543210987_0002,pyspark,idle,Spark UI,Link
